# DO NOT USE - superseded by `panvram_loader_A100_v2.ipynb`

This version fetched the individual `*.q16k.rr3` files from the Zenodo record. Those files are not addressable on
their own there (HTTP 404), so `wget -O` left **0-byte archives** and the five-assembly mode failed twice on Colab.
It is kept only as the record of what was tried. Use `panvram_loader_A100_v2.ipynb`.


# panvram as a window loader for DNA model training: GPU path

The same measurement as the CPU rows of the D1 report, on an A100. One batch of `BATCH` windows of `W` bases,
3 warm-up batches + 9 timed batches, median; the result is a `uint8 [BATCH, W]` tensor already on the card, so no
host-to-device copy is in the number.

What this notebook does **not** do: it does not compare with GenVarLoader or pyfaidx (those are the CPU rows), and
it does not claim the contents match a variant-based loader - panvram returns the assembly's own bases.

Runtime: **A100 (High-RAM)**. The 558-assembly q16k cohort needs about 14 GB of VRAM; the 5-assembly subset about
3.2 GB. Set `N_ASM` below.

Data: the archives come from the Zenodo record of the dataset (DOI 10.5281/zenodo.23232317) and the reference from
DATASET.md of the repository. Downloading the 558 q16k parts is about 8.1 GB; the 5-assembly path is about 70 MB and
is the default here.

In [ ]:
N_ASM = 5          # 5 or 558
BATCH = 1024
WS = (8192, 32768)
RUNS = 9
SEED = 20261003

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!free -g | head -2
!nproc

In [ ]:
# panvram: the published repository, CUDA extension built against Colab's torch
!apt-get -qq install -y libzstd-dev > /dev/null
!git clone -q https://github.com/yasha1971-coder/panvram.git /content/panvram
%cd /content/panvram
!git log --oneline -1
!pip install -q -e . 2>&1 | tail -2
import panvram, torch
print('panvram', panvram.__version__ if hasattr(panvram, '__version__') else 'ok', 'with_cuda', panvram.with_cuda)
print('torch', torch.__version__, torch.cuda.get_device_name(0))
assert panvram.with_cuda, 'the CUDA extension did not build - check nvcc and the torch build'

In [ ]:
# Reference (T2T-CHM13v2.0, DATASET.md: 3 156 259 565 bytes, SHA-256 51ab7d3c...)
%cd /content
!mkdir -p cohort
!wget -q -O t2t.fa.gz https://s3-us-west-2.amazonaws.com/human-pangenomics/T2T/CHM13/assemblies/analysis_set/chm13v2.0.fa.gz
!gunzip -f t2t.fa.gz && mv t2t.fa cohort/reference.fa
!ls -l cohort/reference.fa
!sha256sum cohort/reference.fa

In [ ]:
# Archives. 5 assemblies: the subset5 release asset of the repository (22 763 520 B, SHA-256 ea7f58ef...).
# Those five are v2 prototype archives, so for the v1 loader take the five v1 q16k archives of the Zenodo record
# instead; the cell below fetches the whole q16k tar when N_ASM == 558, else the five files from the record.
ZENODO = 'https://zenodo.org/records/23232317/files'
NAMES5 = ['y1_HG00438.1', 'y1_HG00438.2', 'y1_HG00621.1', 'y1_HG00621.2', 'y1_HG00673.1']
if N_ASM == 558:
    !for i in $(seq -w 0 15); do wget -q -c $ZENODO/cohort558_q16k.tar.part$i; done
    !wget -q -O PARTS.sha256 $ZENODO/PARTS.sha256 && sha256sum -c PARTS.sha256 2>&1 | grep -c ': OK$'
    !cat cohort558_q16k.tar.part?? > cohort558_q16k.tar
    !wget -q -O t.sha $ZENODO/cohort558_q16k.tar.sha256 && sha256sum -c t.sha
    !tar -xf cohort558_q16k.tar -C cohort && rm -f cohort558_q16k.tar cohort558_q16k.tar.part??
    !cd cohort && sha256sum -c SHA256SUMS 2>&1 | grep -c ': OK$'
else:
    # the record ships the archives inside the tar only, so fetch the tar's first parts is wasteful: pull the
    # five files from the cohort directory of the record if present, else fall back to the full tar
    import subprocess
    ok = all(subprocess.run(['wget', '-q', '-O', f'cohort/{n}.q16k.rr3', f'{ZENODO}/{n}.q16k.rr3'],
                            capture_output=True).returncode == 0 for n in NAMES5)
    print('five files fetched directly' if ok else 'direct files not in the record: set N_ASM = 558 and use the tar')
!ls -l cohort | head -8
!du -sh cohort

In [ ]:
import json, time
import torch, panvram

t0 = time.perf_counter()
c = panvram.Cohort.open('/content/cohort', device='cuda', dataset='q16k')
torch.cuda.synchronize()
topen = time.perf_counter() - t0
print(f'OPEN\t{len(c.names)} assemblies\t{topen:.3f} s')
print('resident', torch.cuda.memory_allocated() / 1e9, 'GB allocated;', torch.cuda.max_memory_allocated() / 1e9, 'GB peak')
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

In [ ]:
out = {'tool': 'panvram-v1-q16k', 'device': 'cuda', 'gpu': torch.cuda.get_device_name(0), 'open_s': topen,
       'assemblies': len(c.names), 'batch': BATCH, 'runs': RUNS, 'rows': []}
for W in WS:
    g = torch.Generator(device='cuda').manual_seed(SEED)
    for _ in range(3):
        c.sample(BATCH, W, generator=g)
    torch.cuda.synchronize()
    ts = []
    for _ in range(RUNS):
        gg = torch.Generator(device='cuda').manual_seed(SEED)
        torch.cuda.synchronize()
        a = time.perf_counter()
        x = c.sample(BATCH, W, generator=gg)
        torch.cuda.synchronize()
        ts.append(time.perf_counter() - a)
        assert x.shape == (BATCH, W) and x.is_cuda
    ts.sort()
    med = ts[len(ts) // 2]
    print(f'A-GPU\tW {W}\tbatch {BATCH}\tmedian {med*1e3:.3f} ms\t{BATCH/med:.0f} windows/s\t{BATCH*W/med/1e9:.3f} GB/s\tmin {min(ts)*1e3:.3f}\tmax {max(ts)*1e3:.3f}')
    out['rows'].append({'W': W, 'median_s': med, 'min_s': min(ts), 'max_s': max(ts),
                        'windows_per_s': BATCH / med, 'gb_per_s': BATCH * W / med / 1e9})
print(json.dumps(out, indent=1))

In [ ]:
# Correctness on the card: every window of one batch equals the CPU decoder's answer for the same coordinates.
W = WS[0]
g = torch.Generator(device='cuda').manual_seed(SEED)
x, coords = c.sample(256, W, generator=g, return_coords=True)
cc = panvram.Cohort.open('/content/cohort', device='cpu', dataset='q16k')
bad = 0
for i in range(coords.shape[0]):
    a, ci, s, _ = [int(v) for v in coords[i]]
    y = cc.fetch(cc.names[a], cc._core.contigs(a)[ci][0], s, W)
    if not torch.equal(x[i].cpu(), y):
        bad += 1
print(f'CHECK\tW {W}\twindows {coords.shape[0]}\tidentical {coords.shape[0]-bad}\tdiffer {bad}')